# Clasificación de datos de los sujetos

En el presente archivo, se muestra el entrenamiento de los clasificadores de datos del laboratorio 2 del módulo tres del curso 1MTR53 de la PUCP

In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.tree import DecisionTreeClassifier

DATA_DIR = Path("data")
FS = 256

EOG_PATHS = [DATA_DIR / "S1-EOG.csv",
            DATA_DIR / "S2-EOG.csv",
            DATA_DIR / "S3-EOG.csv",
            DATA_DIR / "S4-EOG.csv",
            DATA_DIR / "S5-EOG.csv",
            DATA_DIR / "S6-EOG.csv",
            DATA_DIR / "S7-EOG.csv"]

CS_PATHS = [DATA_DIR / "S1-ControlSignal.csv",
            DATA_DIR / "S2-ControlSignal.csv",
            DATA_DIR / "S3-ControlSignal.csv",
            DATA_DIR / "S4-ControlSignal.csv",
            DATA_DIR / "S5-ControlSignal.csv",
            DATA_DIR / "S6-ControlSignal.csv",
            DATA_DIR / "S7-ControlSignal.csv"]

VALID_LABELS = {1, 2, 3}
EOG_COLUMNS = ["V1", "V2", "V3", "V4"]

## Lectura y preparación de datos
Para una mayor facilidad, se juntan los datos en una lista. El label subject_id se añade para hacer la diferenciación de datos.

In [91]:
def load_data(eog_paths, cs_paths):
    all_dfs = []
    
    for subject_id, (eog_path, cs_path) in enumerate(zip(eog_paths, cs_paths), start=1):
        df_eog = pd.read_csv(eog_path)
        df_cs = pd.read_csv(cs_path)
        # print(df_cs)
        
        # Ajustar nombres de columnas si es necesario
        df_eog.columns = ['V1', 'V2', 'V3', 'V4']
        target_col = df_cs.columns[0]
        
        df_sub = df_eog.copy()
        df_sub['target'] = df_cs[target_col].values
        df_sub['subject_id'] = subject_id
        
        all_dfs.append(df_sub)
        
    df_full = pd.concat(all_dfs, ignore_index=True)
    
    # Eliminar filas con valores nulos o clases no válidas
    df_full = df_full.dropna()
    df_full = df_full[df_full['target'].isin([1, 2, 3])]
    
    return df_full

df = load_data(EOG_PATHS, CS_PATHS)
print(df.shape)

(2196125, 6)


# Extracción de características
Ya con los datos preparados, se procede a extraer sus características por ventanas de tiempo

In [96]:
WINDOW_SEC = 1    # tamaño de ventana en segundos
OVERLAP = 0.75      # fracción de overlap: valor entre 0 y 1

WIN_SIZE = int(WINDOW_SEC * FS)
STEP = int(WIN_SIZE * (1 - OVERLAP))

def extract_features(df):
    X_list, y_list, groups_list = [], [], []
    
    # Lista con el orden exacto de las 16 características
    feature_names = [
        'EOGh_mean', 'EOGh_std', 'EOGh_ptp', 'EOGh_rms', 'EOGh_mav', 'EOGh_wl', 'EOGh_max', 'EOGh_min',
        'EOGv_mean', 'EOGv_std', 'EOGv_ptp', 'EOGv_rms', 'EOGv_mav', 'EOGv_wl', 'EOGv_max', 'EOGv_min'
    ]
    
    for subject_id, group in df.groupby('subject_id'):
        v1, v2 = group['V1'].values, group['V2'].values
        v3, v4 = group['V3'].values, group['V4'].values
        targets = group['target'].values
        
        # Canales derivados EOG horizontal y vertical
        eogh = v1 - v2
        eogv = v3 - v4
        
        for start in range(0, len(group) - WIN_SIZE + 1, STEP):
            end = start + WIN_SIZE
            
            win_h = eogh[start:end]
            win_v = eogv[start:end]
            win_target = targets[start:end]
            
            mode_target = pd.Series(win_target).mode()[0]
            
            # --- CARACTERÍSTICAS EOG HORIZONTAL (EOGh) ---
            eogh_mean = np.mean(win_h)
            eogh_std  = np.std(win_h)      
            eogh_ptp  = np.ptp(win_h)
            eogh_rms  = np.sqrt(np.mean(win_h**2))
            eogh_mav  = np.mean(np.abs(win_h))
            eogh_wl   = np.sum(np.abs(np.diff(win_h)))
            eogh_max  = np.max(win_h)
            eogh_min  = np.min(win_h)
            
            # --- CARACTERÍSTICAS EOG VERTICAL (EOGv) ---
            eogv_mean = np.mean(win_v)
            eogv_std  = np.std(win_v)
            eogv_ptp  = np.ptp(win_v)
            eogv_rms  = np.sqrt(np.mean(win_v**2))
            eogv_mav  = np.mean(np.abs(win_v))
            eogv_wl   = np.sum(np.abs(np.diff(win_v)))
            eogv_max  = np.max(win_v)
            eogv_min  = np.min(win_v)
            
            feats = [
                eogh_mean, eogh_std, eogh_ptp, eogh_rms, eogh_mav, eogh_wl, eogh_max, eogh_min,
                eogv_mean, eogv_std, eogv_ptp, eogv_rms, eogv_mav, eogv_wl, eogv_max, eogv_min
            ]
            
            X_list.append(feats)
            y_list.append(mode_target)
            groups_list.append(subject_id)
            
    df_features = pd.DataFrame(X_list, columns=feature_names)
    df_features['target'] = y_list
    df_features['subject_id'] = groups_list
    
    return df_features

df_feats = extract_features(df)

feature_cols = [c for c in df_feats.columns if c not in ['target', 'subject_id']]
X = df_feats[feature_cols].values
y = df_feats['target'].values
groups = df_feats['subject_id'].values

## Entrenamiento
Se decide separar los datos correspondientes en train y test. El sujeto 7 será reservado para los datos de testeo.

Se entrenarán 2 modelos
- Random Forest
- Super Vector Machine (SVM)

In [97]:
TEST_SUBJECT_ID = 7  # Identificación del sujeto reservado para prueba

# Máscaras booleanas según el sujeto
train_mask = (groups != TEST_SUBJECT_ID)
test_mask = (groups == TEST_SUBJECT_ID)

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Normalización / Escalado (Ajustado únicamente con datos de entrenamiento)
#scaler = StandardScaler()
#X_train_scaled = scaler.fit_transform(X_train)
#X_test_scaled = scaler.transform(X_test)

In [98]:
RANDOM_STATE = 42

models = {
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_split=5, min_samples_leaf=5),
    "Support Vector Machine (SVM)": SVC(kernel='rbf', C=1.0, gamma="scale"),
    "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=5,min_samples_split=10, min_samples_leaf=5),
    "Random Forest 2": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),

}

results = []

for name, model in models.items():
    # Entrenamiento
    model.fit(X_train, y_train)
    
    # Predicción en el sujeto de prueba (Sujeto 7)
    y_pred = model.predict(X_test)
    
    # Cálculo de métricas de evaluación
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        "Modelo": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

## Resultados

In [99]:
df_results = pd.DataFrame(results)

print("="*60)
print(f"SUJETO RESERVADO PARA PRUEBA: Sujeto {TEST_SUBJECT_ID}")
print("Sujetos de entrenamiento: Sujetos 1, 2, 3, 4, 5, 6")
print("="*60)
print("\n--- TABLA COMPARATIVA DE MODELOS ---")
print(df_results.to_string(index=False))
print("="*60)

SUJETO RESERVADO PARA PRUEBA: Sujeto 7
Sujetos de entrenamiento: Sujetos 1, 2, 3, 4, 5, 6

--- TABLA COMPARATIVA DE MODELOS ---
                      Modelo  Accuracy  Precision  Recall  F1-Score
               Random Forest    0.6160     0.6728  0.6160    0.5524
Support Vector Machine (SVM)    0.6353     0.6989  0.6353    0.5498
               Decision Tree    0.6445     0.6797  0.6445    0.5877
             Random Forest 2    0.5527     0.6236  0.5527    0.5133
